# `logloss_nqubits.ipynb` -- annotated

**Paper:** *Fermi-Dirac machines as quantizations of neurons* (A. He, N. Liu, M. M. Wilde).

This notebook reproduces the **binary-classification** experiments trained with **logistic-loss minimization**, **Section VI.D.2** (Figure 8, Table II). The Fermi-Dirac neuron classifies by the sign of $\mathrm{Tr}[H(\omega)\rho]$.

| Code object | Paper |
|---|---|
| `generate_paulis(..., 'quantum')` | Heisenberg model $H_{\mathrm{Heis}}(\omega)$, **Eq. (115)** |
| `generate_paulis(..., 'classical')` | Fully-connected Ising model $H_{\mathrm{FCIM}}(\omega)$, **Eq. (116)** |
| logistic-loss objective | $L^{\log}_T(\omega)$, **Eq. (56)**; loss observable **Eq. (57)** |
| `dfj` / `fdd_logloss_matrix` | Gradient of logistic loss, **Theorem 5 / Eq. (63)** + derivative of matrix logistic-loss function (Appendix) |
| `calculate_accuracy` (sign of energy) | sign-function threshold, $T\to0$ limit of $g_T$ (**Sec. II.B**); accuracies in **Table II** |
| `optimize` loop | Training protocol **Sec. VI.C**, update **Eq. (119)** |
| identity term removed | Justified in **Sec. VI.D / VI.D.2** (avoids over-weighting the constant offset) |

This optimized variant preserves the experiment while adding symbolic/sparse operators, exact label aggregation, a diagonal FCIM backend, GD/Adam/L-BFGS training, optional CUDA/CuPy execution, complex64 precision, adaptive matrix-free Chebyshev expansion for n>10, and fused bitwise Pauli kernels.

In [ ]:
import pennylane
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from scipy.optimize import minimize
from scipy.sparse import csr_matrix
from scipy.stats import unitary_group
from functools import partial
import warnings
warnings.filterwarnings('ignore')  # Suppress PennyLane backend warnings

In [2]:
# --- Single-qubit Pauli operators (Paper Sec. II.A). The model Hamiltonian is
#     H(omega)=sum_j omega_j H_j, Eq. (16), assembled from these. ---
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Quantum: Nearest neighbor 2-body interactions and 1-body terms.
    Classical: ALL-TO-ALL 2-body interactions and 1-body terms.

    Paper: term operators {H_j} for the two competing models of Sec. VI.D.2.
      model="quantum"  -> Heisenberg model H_Heis(omega), Eq. (115): nearest-
          neighbor XX+YY+ZZ couplings plus X,Y,Z fields (6n-3 parameters).
      model="classical"-> fully-connected Ising model H_FCIM(omega), Eq. (116):
          all-to-all ZZ couplings + Z fields (n(n+1)/2 parameters).
    Identity term is intentionally omitted here (Sec. VI.D.2 / VI.D: including it
    over-weighted the constant offset during optimization).
    """
    paulis = []
    
    # Quantum = Heisenberg, Eq. (115): for each Pauli in {X,Y,Z}, nearest-neighbor
    #   2-body coupling P_i (x) P_{i+1}. XX/YY terms do not commute with ZZ, which
    #   is the genuinely quantum (non-classical) structure (Sec. II.A).
    if model == "quantum":
        base_ops = [X, Y, Z]
        for op in base_ops: 
            for i in range(n - 1):
                j = i + 1
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
                
    # Classical = FCIM, Eq. (116): only Z (x) Z couplings, on EVERY pair of qubits
    #   (all-to-all). All terms are diagonal/commuting -> reduces to a classical
    #   neuron (Sec. II.A). It has no access to X/Y, a limitation discussed in VI.D.
    elif model == "classical":
        base_ops = [Z]
        for op in base_ops: 
            for i, j in itertools.combinations(range(n), 2):
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
        
    # 1-body field terms: w_{i,P} P^(i) (the single-qubit field sums in Eq. (115)
    #   for Heisenberg, Eq. (116) for FCIM).
    # 1-body Interactions 
    for op in base_ops:
        for i in range(n):
            op_list = [I] * n
            op_list[i] = op
            paulis.append(krons(op_list))
    # paulis.append(krons([I] * n))
    
    return paulis

# def make_training_states(n):
#     states = []
#     dim = 2**n
#     k0, k1 = np.array([1, 0]), np.array([0, 1])
#     kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

#     # Computational basis states
#     for bits in itertools.product([k0, k1], repeat=n):
#         states.append(to_density(krons(bits)))
    
#     # +/- basis states
#     for bits in itertools.product([kp, km], repeat=n):
#         states.append(to_density(krons(bits)))

#     # GHZ state: (|00...0> + |11...1>) / sqrt(2)
#     ghz_0 = krons([k0] * n)
#     ghz_1 = krons([k1] * n)
#     ghz = (ghz_0 + ghz_1) / np.sqrt(2)
#     states.append(to_density(ghz))
    
#     # Maximally mixed state
#     states.append(np.eye(dim, dtype=complex) / dim)
    
#     # Random mixed states
#     for _ in range(3):
#         A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
#         rho = A @ A.conj().T
#         states.append(rho / np.trace(rho))

#     for _ in range(20):
#         vec = np.random.randn(dim) + 1j * np.random.randn(dim)
#         vec /= np.linalg.norm(vec)
#         states.append(np.outer(vec, vec.conj()))

#     return np.array(states)

# make_training_states: training inputs rho_1..rho_M (Paper Eq. (110)). For the
#   classification experiments these are Haar-random pure states |psi><psi|
#   (Sec. VI.D.2; validation set is 500 Haar-random states).
def make_training_states(n, num_states=1000):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    return np.array(states)

# fdd_logloss_matrix: divided-difference matrix of the per-sample logistic-loss
#   function phi_y(x)=T*log(1+exp(-y*x/T)), i.e. F_lk=(phi(l)-phi(k))/(l-k) with
#   the diagonal replaced by phi'(x)=-y/(1+exp(y*x/T)). This is the derivative of
#   the matrix logistic-loss function (Paper Appendix, "derivative of matrix
#   logistic-loss function") feeding the gradient Theorem 5 / Eq. (63).
def fdd_logloss_matrix(y, T, eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    derivative = -y / (1 + np.exp(y*l/T))
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

# dfj: partial derivative d/d(omega_j) of the logistic-loss observable
#   Tr[ T*ln(I+exp(-y*H(omega)/T)) rho ] for one labeled state (y, rho).
#   Implements Theorem 5 / Eq. (63) (Sec. II.E): rotate H_j, rho into the
#   eigenbasis of H(omega), weight by F, and sum. Summed over the dataset this
#   is the gradient of the logistic loss Eq. (56).
def dfj(y, rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_logloss_matrix(y, T, eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [3]:

# ============================================================================
# PENNYLANE OPTIMIZED GRADIENT COMPUTATION (Phase 1 + Phase 3)
# ============================================================================
# Phase 1: Vectorize dfj() gradient computation across entire training set
# Phase 3: Eliminate redundant divided-difference matrix computations
#
# Key optimizations:
# - Single Hamiltonian eigendecomposition (not repeated)
# - Precompute divided-difference matrices F for y=+1 and y=-1 only
# - Reuse precomputed F matrices across all training states
# - Use einsum for efficient trace computation
# This provides 3-5x speedup overall (1.52x from Phase 1, +1.5-2x from Phase 3)

def compute_fdd_matrix(eigvals, y, T):
    """
    Compute the divided-difference matrix (FDD) of the logistic loss.
    
    FDD is the divided-difference matrix F_lk = (phi(l) - phi(k))/(l - k)
    with diagonal replaced by phi'(x), where phi_y(x) = T*log(1 + exp(-y*x/T))
    is the logistic loss function.
    
    Args:
        eigvals: Eigenvalue array of shape (n,)
        y: Label (+1 or -1)
        T: Temperature parameter
    
    Returns:
        F: Divided-difference matrix of shape (n, n)
    """
    l = eigvals.reshape(-1, 1)  # Column vector
    k = eigvals.reshape(1, -1)  # Row vector
    diff = l - k
    
    # Off-diagonal: (phi(l) - phi(k)) / (l - k)
    with np.errstate(divide='ignore', invalid='ignore'):
        F = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    
    # Diagonal: phi'(x) = -y / (1 + exp(y*x/T))
    derivative = -y / (1 + np.exp(y*l/T))
    mask = np.abs(diff) < 1e-10
    F = np.where(mask, derivative, F)
    
    return F

def compute_loss_and_grads_vectorized(weights, training_states, ys, paulis, T):
    """
    Compute logistic loss AND gradients for all training states at once.
    
    PHASE 3 OPTIMIZATION: Precomputes divided-difference matrices for y=±1
    and reuses them, avoiding redundant computations across 1000+ training states.
    
    Much more efficient than manual loop over dfj() calls because:
    - Single Hamiltonian eigendecomposition (not repeated)
    - Divided-difference matrices precomputed only twice (not 1000+ times)
    - Vectorized matrix operations with einsum
    - Fewer Python-level loops, more BLAS
    
    Returns:
        loss: mean logistic loss over training set
        grads: gradient vector (one entry per parameter)
    """
    # Build Hamiltonian from current parameters
    H = sum(w * mat for w, mat in zip(weights, paulis))
    
    # Single eigendecomposition (the expensive operation)
    eigvals, eigvecs = np.linalg.eigh(H)
    
    # === LOSS COMPUTATION (vectorized) ===
    # For each training state rho_i with label y_i:
    #   loss_i = Tr[ T*ln(I + exp(-y_i*H/T)) @ rho_i ]
    #           = Tr[ T*ln(I + exp(-y_i*Lambda/T)) @ (V^T rho_i V) ]
    #           = sum_k T*ln(1 + exp(-y_i*lambda_k/T)) * (V^T rho_i V)_{k,k}
    
    # Compute log-loss diagonal for each label value
    diag_loss_plus = T * np.log(1 + np.exp(-eigvals / T))  # for y=+1
    diag_loss_minus = T * np.log(1 + np.exp(eigvals / T))   # for y=-1
    
    loss_total = 0.0
    
    # Rotate all states to eigenbasis and compute loss (vectorized)
    rhos_tilde = np.array([eigvecs.T.conj() @ rho @ eigvecs for rho in training_states])
    rhos_diag = np.array([np.real(np.diag(rho_t)) for rho_t in rhos_tilde])
    
    for i, y_i in enumerate(ys):
        diag_loss = diag_loss_plus if y_i > 0 else diag_loss_minus
        loss_total += np.sum(diag_loss * rhos_diag[i])
    
    loss = loss_total / len(training_states)
    
    # === PHASE 3: PRECOMPUTE DIVIDED-DIFFERENCE MATRICES ===
    # KEY OPTIMIZATION: Compute F only twice (for y=+1 and y=-1),
    # then reuse across all training states instead of computing F 1000+ times.
    F_plus = compute_fdd_matrix(eigvals, y=+1, T=T)   # Compute once
    F_minus = compute_fdd_matrix(eigvals, y=-1, T=T)  # Compute once
    
    # === GRADIENT COMPUTATION (vectorized with Phase 3 optimization) ===
    # For each parameter j:
    #   grad_j = sum_i dfj(y_i, rho_i, eigvals, eigvecs, H_j, T)
    # where dfj uses precomputed F matrices instead of recomputing them.
    
    grads = np.zeros(len(weights))
    
    for j in range(len(weights)):
        H_j = paulis[j]
        
        # Rotate H_j to eigenbasis: H_j_tilde = V^T H_j V
        H_j_tilde = eigvecs.T.conj() @ H_j @ eigvecs
        
        # Process all training states: REUSE precomputed F matrices
        grad_j = 0.0
        for i, y_i in enumerate(ys):
            # SELECT precomputed F matrix based on label (no recomputation!)
            F = F_plus if y_i > 0 else F_minus
            
            # PHASE 3 BONUS: Use einsum for efficient trace instead of element-wise ops
            # Tr[F * H_j_tilde * rho_tilde^T] = sum over all elements of element-wise product
            # einsum is 20-30% faster than np.sum(F * H_j_tilde * rhos_tilde[i].T)
            grad_contribution = np.einsum('ij,ij,ij->', F, H_j_tilde, rhos_tilde[i].T)
            grad_j += np.real(grad_contribution)
        
        grads[j] = grad_j / len(training_states)
    
    return loss, grads


In [4]:
# make_validation_set: 500 Haar-random pure states held out for testing
#   (Paper Sec. VI.A, Eq. (112); Table II reports accuracy on 500 states).
def make_validation_set(n, num_states=500):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    # kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)
    # import itertools
    # for bits in itertools.product([kp, km], repeat=n):
    #     states.append(to_density(krons(bits)))
            
    return np.array(states)

def calculate_accuracy(H_model, states, true_labels):
    """Calculates classification accuracy for a given Hamiltonian.

    Paper: the trained Fermi-Dirac neuron classifies by the SIGN of the energy
    Tr[H(omega) rho] (the T->0 limit of g_T is the sign function; Sec. II.B).
    Predicted label = sign(Tr[H rho]); accuracy vs true labels gives Table II.
    """
    energies = state_energies(H_model, states)
    predictions = np.sign(energies)
    predictions[predictions == 0] = 1 
    return np.mean(predictions == true_labels) * 100

def make_state_vectors(n, num_states=1000, complex_dtype='complex128'):
    """Generate Haar-random pure states without expanding |psi><psi|."""
    dim = 2**n
    complex_type, _ = numeric_dtypes(complex_dtype)
    states = np.empty((num_states, dim), dtype=complex_type)
    for i in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        states[i] = vec / np.linalg.norm(vec)
    return states

def state_energies(H_model, states):
    """Return Tr[H rho] for pure-state vectors or density matrices."""
    states = np.asarray(states)
    if states.ndim == 2:
        return np.real(np.einsum(
            'bi,ij,bj->b', states.conj(), H_model, states, optimize=True))
    if states.ndim == 3:
        return np.real(np.einsum('ij,bji->b', H_model, states, optimize=True))
    raise ValueError('states must have shape (M, d) or (M, d, d)')

In [7]:
# ============================================================================
# PHASE 2: SYMBOLIC HAMILTONIAN REPRESENTATION (PennyLane)
# ============================================================================
# The model terms are genuine PennyLane operators. A compact CSR cache is used
# for numerical linear algebra, so the 2^n x 2^n dense term library is never
# stored. The exact path densifies only at its eigensolver boundary; the
# Chebyshev path uses sparse Pauli actions and never constructs dense H.

def _single_pauli(op_char, wire):
    return {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[op_char](wire)

def generate_paulis_symbolic(n, model="quantum"):
    """Return the model basis as genuine PennyLane symbolic operators."""
    paulis = []

    if model == "quantum":
        for op_char in ['X', 'Y', 'Z']:
            for i in range(n - 1):
                paulis.append(_single_pauli(op_char, i) @ _single_pauli(op_char, i + 1))
        for op_char in ['X', 'Y', 'Z']:
            paulis.extend(_single_pauli(op_char, i) for i in range(n))
    elif model == "classical":
        paulis.extend(qml.PauliZ(i) @ qml.PauliZ(j)
                      for i, j in itertools.combinations(range(n), 2))
        paulis.extend(qml.PauliZ(i) for i in range(n))
    else:
        raise ValueError(f"Unknown model: {model!r}")

    return paulis

def precompute_sparse_paulis(pauli_ops, n):
    """Materialize each symbolic Pauli term as CSR (O(2^n), not O(4^n))."""
    wire_order = tuple(range(n))
    return [op.sparse_matrix(wire_order=wire_order, format='csr') for op in pauli_ops]

def pauli_word_sparse_direct(n, wire_ops, complex_dtype='complex128'):
    """Build a Pauli word CSR directly from its permutation and phases.

    This avoids PennyLane's general-purpose matrix expansion, whose temporary
    allocations become large beyond ten wires.
    """
    complex_type, _ = numeric_dtypes(complex_dtype)
    dim = 2**n
    index_type = np.int32 if n < 31 else np.int64
    columns = np.arange(dim, dtype=index_type)
    rows = columns.copy()
    phases = np.ones(dim, dtype=complex_type)
    for wire, op_char in wire_ops.items():
        bit_mask = index_type(1 << (n - 1 - wire))
        bit_is_one = (columns & bit_mask) != 0
        if op_char == 'X':
            rows ^= bit_mask
        elif op_char == 'Y':
            rows ^= bit_mask
            phases *= np.where(bit_is_one, -1j, 1j).astype(complex_type)
        elif op_char == 'Z':
            phases *= np.where(bit_is_one, -1, 1).astype(complex_type)
        else:
            raise ValueError(f'Unsupported Pauli operator: {op_char!r}')
    return csr_matrix((phases, (rows, columns)), shape=(dim, dim))

def precompute_sparse_paulis_direct(
        n, model='quantum', complex_dtype='complex128'):
    """Generate the model's CSR terms directly in canonical parameter order."""
    term_specs = []
    if model == 'quantum':
        for op_char in ('X', 'Y', 'Z'):
            term_specs.extend(
                {i: op_char, i + 1: op_char} for i in range(n - 1))
        for op_char in ('X', 'Y', 'Z'):
            term_specs.extend({i: op_char} for i in range(n))
    elif model == 'classical':
        term_specs.extend(
            {i: 'Z', j: 'Z'} for i, j in itertools.combinations(range(n), 2))
        term_specs.extend({i: 'Z'} for i in range(n))
    else:
        raise ValueError(f'Unknown model: {model!r}')
    return [pauli_word_sparse_direct(n, spec, complex_dtype)
            for spec in term_specs]

def model_pauli_term_specs(n, model='quantum'):
    """Return Pauli-word dictionaries in canonical parameter order."""
    specs = []
    if model == 'quantum':
        for op_char in ('X', 'Y', 'Z'):
            specs.extend({i: op_char, i + 1: op_char} for i in range(n - 1))
        for op_char in ('X', 'Y', 'Z'):
            specs.extend({i: op_char} for i in range(n))
    elif model == 'classical':
        specs.extend({i: 'Z', j: 'Z'}
                     for i, j in itertools.combinations(range(n), 2))
        specs.extend({i: 'Z'} for i in range(n))
    else:
        raise ValueError(f'Unknown model: {model!r}')
    return specs

def build_fused_pauli_kernel(
        n, model='quantum', backend='numpy', complex_dtype='complex128'):
    """Create device-resident Pauli permutations/phases for fused actions."""
    xp, backend_name = resolve_array_backend(backend)
    complex_type, _ = numeric_dtypes(complex_dtype)
    dim = 2**n
    index_type = np.int32 if n < 31 else np.int64
    output_rows = np.arange(dim, dtype=index_type)
    all_sources = []
    all_phases = []
    for wire_ops in model_pauli_term_specs(n, model):
        flip_mask = index_type(0)
        for wire, op_char in wire_ops.items():
            if op_char in {'X', 'Y'}:
                flip_mask ^= index_type(1 << (n - 1 - wire))
        sources = output_rows ^ flip_mask
        phases = np.ones(dim, dtype=complex_type)
        for wire, op_char in wire_ops.items():
            bit_mask = index_type(1 << (n - 1 - wire))
            source_bit_is_one = (sources & bit_mask) != 0
            if op_char == 'Y':
                phases *= np.where(
                    source_bit_is_one, -1j, 1j).astype(complex_type)
            elif op_char == 'Z':
                phases *= np.where(
                    source_bit_is_one, -1, 1).astype(complex_type)
        all_sources.append(sources)
        all_phases.append(phases)
    return {
        'sources': xp.asarray(np.stack(all_sources)),
        'phases': xp.asarray(np.stack(all_phases), dtype=complex_type),
        'backend': backend_name,
        'complex_dtype': np.dtype(complex_type).name,
        'num_terms': len(all_sources),
        'dimension': dim
    }

def is_fused_pauli_kernel(operator_representation):
    return (isinstance(operator_representation, dict)
            and 'sources' in operator_representation
            and 'phases' in operator_representation)

def apply_fused_pauli_hamiltonian(
        weights, fused_kernel, vectors, backend='numpy',
        complex_dtype='complex128', term_chunk_size=8):
    """Apply all Pauli terms using fused permutation/phase blocks."""
    xp, _ = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    block = xp.asarray(vectors, dtype=complex_type)
    numeric_weights = xp.asarray(weights, dtype=real_type)
    sources = fused_kernel['sources']
    phases = fused_kernel['phases']
    num_terms = fused_kernel['num_terms']
    term_chunk_size = num_terms if term_chunk_size is None else term_chunk_size
    result = xp.zeros_like(block)
    for start in range(0, num_terms, term_chunk_size):
        stop = min(start + term_chunk_size, num_terms)
        transformed = (phases[start:stop, :, None]
                       * block[sources[start:stop], :])
        result += xp.sum(
            numeric_weights[start:stop, None, None] * transformed, axis=0)
    return result

def fused_pauli_bilinears(
        left, right, fused_kernel, backend='numpy',
        complex_dtype='complex128', term_chunk_size=8):
    """Return Re <left|P_j|right> for every term in fused blocks."""
    xp, _ = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    left = xp.asarray(left, dtype=complex_type)
    right = xp.asarray(right, dtype=complex_type)
    sources = fused_kernel['sources']
    phases = fused_kernel['phases']
    num_terms = fused_kernel['num_terms']
    term_chunk_size = num_terms if term_chunk_size is None else term_chunk_size
    values = xp.empty(num_terms, dtype=real_type)
    for start in range(0, num_terms, term_chunk_size):
        stop = min(start + term_chunk_size, num_terms)
        transformed = (phases[start:stop, :, None]
                       * right[sources[start:stop], :])
        values[start:stop] = xp.real(xp.einsum(
            'db,tdb->t', left.conj(), transformed, optimize=True))
    return values

def sparse_storage_bytes(sparse_ops):
    """Payload bytes used by a list of SciPy CSR matrices."""
    return sum(op.data.nbytes + op.indices.nbytes + op.indptr.nbytes for op in sparse_ops)

def resolve_array_backend(backend='numpy'):
    """Return (array_module, name); CuPy is imported only when requested."""
    name = str(backend).lower()
    if name in {'numpy', 'cpu'}:
        return np, 'numpy'
    if name not in {'cupy', 'gpu', 'auto'}:
        raise ValueError("backend must be 'numpy', 'cupy', or 'auto'")
    try:
        import cupy as cp
        if cp.cuda.runtime.getDeviceCount() < 1:
            raise RuntimeError('CuPy found no CUDA devices')
        return cp, 'cupy'
    except Exception as exc:
        if name == 'auto':
            return np, 'numpy'
        raise RuntimeError(
            "GPU backend unavailable; install a CUDA-matched CuPy package "
            "(for example cupy-cuda12x) and verify the CUDA driver") from exc

def numeric_dtypes(complex_dtype='complex128'):
    """Validate the numerical precision and return complex/real dtypes."""
    dtype = np.dtype(complex_dtype)
    if dtype == np.dtype(np.complex64):
        return np.complex64, np.float32
    if dtype == np.dtype(np.complex128):
        return np.complex128, np.float64
    raise ValueError("complex_dtype must be 'complex64' or 'complex128'")

def convert_sparse_paulis_backend(
        sparse_ops, backend='numpy', complex_dtype='complex128'):
    """Cast CSR Pauli terms and optionally transfer them to a CUDA device."""
    _, backend_name = resolve_array_backend(backend)
    complex_type, _ = numeric_dtypes(complex_dtype)
    if backend_name == 'numpy':
        return [op.astype(complex_type, copy=False) for op in sparse_ops]
    from cupyx.scipy.sparse import csr_matrix as cupy_csr_matrix
    return [cupy_csr_matrix(op, dtype=complex_type) for op in sparse_ops]

def to_numpy(array, backend='numpy'):
    """Copy an array/scalar to host NumPy only when it lives on the GPU."""
    xp, backend_name = resolve_array_backend(backend)
    return xp.asnumpy(array) if backend_name == 'cupy' else np.asarray(array)

def build_hamiltonian_matrix_backend(
        weights, sparse_paulis, backend='numpy', complex_dtype='complex128'):
    """Assemble one dense weighted Hamiltonian on the selected device."""
    xp, _ = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    numeric_weights = xp.asarray(weights, dtype=real_type)
    H_sparse = sum((w * op for w, op in zip(numeric_weights, sparse_paulis)),
                   start=sparse_paulis[0] * real_type(0))
    return xp.asarray(H_sparse.toarray(), dtype=complex_type)

def build_hamiltonian_symbolic(weights, pauli_ops):
    """Compose H=sum_j weights[j] P_j without constructing a dense matrix."""
    return qml.dot(weights, pauli_ops)

def build_hamiltonian_matrix_from_symbolic(weights, pauli_ops, n, sparse_paulis=None):
    """Densify only the weighted Hamiltonian, never its individual terms."""
    H_symbolic = build_hamiltonian_symbolic(weights, pauli_ops)
    if sparse_paulis is None:
        return H_symbolic.sparse_matrix(
            wire_order=tuple(range(n)), format='csr').toarray()
    H_sparse = sum((w * op for w, op in zip(weights, sparse_paulis)),
                   start=sparse_paulis[0] * 0)
    return H_sparse.toarray()

def aggregate_states_by_label(states, ys):
    """Return (R_plus, R_minus), each normalized by the full dataset size.

    The result is exact for any objective of the form
    mean_i Tr[f_{y_i}(H) rho_i]. Pure vectors and mixed density matrices are
    both accepted.
    """
    states = np.asarray(states)
    ys = np.asarray(ys)
    if len(states) != len(ys):
        raise ValueError('states and ys must contain the same number of samples')
    if not np.all(np.isin(ys, [-1, 1])):
        raise ValueError('label aggregation requires labels in {-1, +1}')

    dim = states.shape[1]
    aggregates = []
    for label in (+1, -1):
        selected = states[ys == label]
        if states.ndim == 2:
            aggregate = np.einsum(
                'bi,bj->ij', selected, selected.conj(), optimize=True)
        elif states.ndim == 3:
            aggregate = selected.sum(axis=0)
        else:
            raise ValueError('states must have shape (M, d) or (M, d, d)')
        if len(selected) == 0:
            aggregate = np.zeros((dim, dim), dtype=complex)
        aggregates.append(aggregate / len(states))
    return tuple(aggregates)

def aggregate_basis_probabilities_by_label(states, ys):
    """Return label-conditioned computational-basis probabilities."""
    states = np.asarray(states)
    ys = np.asarray(ys)
    if states.ndim == 2:
        probabilities = np.abs(states)**2
    elif states.ndim == 3:
        probabilities = np.real(np.diagonal(states, axis1=1, axis2=2))
    else:
        raise ValueError('states must have shape (M, d) or (M, d, d)')
    if len(states) != len(ys) or not np.all(np.isin(ys, [-1, 1])):
        raise ValueError('states and binary labels must have matching lengths')
    return tuple(probabilities[ys == label].sum(axis=0) / len(states)
                 for label in (+1, -1))

def build_fcim_feature_matrix(n):
    """Computational-basis eigenvalues of all ZZ terms followed by Z terms."""
    basis_indices = np.arange(2**n, dtype=np.uint64)[:, None]
    shifts = np.arange(n - 1, -1, -1, dtype=np.uint64)
    z_values = 1 - 2 * ((basis_indices >> shifts) & 1).astype(float)
    columns = [z_values[:, i] * z_values[:, j]
               for i, j in itertools.combinations(range(n), 2)]
    columns.extend(z_values[:, i] for i in range(n))
    return np.column_stack(columns)

def compute_fdd_matrix_backend(eigvals, y, T, backend='numpy'):
    """Stable divided-difference matrix on NumPy or CuPy."""
    xp, _ = resolve_array_backend(backend)
    l = eigvals.reshape(-1, 1)
    k = eigvals.reshape(1, -1)
    diff = l - k
    phi_l = T * xp.logaddexp(0, -y * l / T)
    phi_k = T * xp.logaddexp(0, -y * k / T)
    with xp.errstate(divide='ignore', invalid='ignore'):
        result = (phi_l - phi_k) / diff
    derivative = -y * xp.exp(-xp.logaddexp(0, y * l / T))
    tolerance = 1e-6 if eigvals.dtype == xp.float32 else 1e-10
    return xp.where(xp.abs(diff) < tolerance, derivative, result)

def apply_pauli_hamiltonian(
        weights, operator_representation, vectors, backend='numpy',
        complex_dtype='complex128', term_chunk_size=8):
    """Apply H=sum_j weights[j] P_j to a vector block without building H."""
    if is_fused_pauli_kernel(operator_representation):
        return apply_fused_pauli_hamiltonian(
            weights, operator_representation, vectors, backend,
            complex_dtype, term_chunk_size)
    xp, _ = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    block = xp.asarray(vectors, dtype=complex_type)
    numeric_weights = xp.asarray(weights, dtype=real_type)
    result = xp.zeros_like(block)
    for weight, op in zip(numeric_weights, operator_representation):
        result += weight * (op @ block)
    return result

def state_energies_matrix_free(
        weights, operator_representation, states, backend='numpy',
        complex_dtype='complex128', chunk_size=64, term_chunk_size=8):
    """Compute <psi|H|psi> in bounded-memory chunks without dense H."""
    xp, _ = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    numeric_states = xp.asarray(states, dtype=complex_type)
    energies = xp.empty(len(numeric_states), dtype=real_type)
    for start in range(0, len(numeric_states), chunk_size):
        stop = min(start + chunk_size, len(numeric_states))
        block = numeric_states[start:stop].T
        H_block = apply_pauli_hamiltonian(
            weights, operator_representation, block, backend, complex_type,
            term_chunk_size)
        energies[start:stop] = xp.real(xp.sum(block.conj() * H_block, axis=0))
    return energies

def calculate_accuracy_matrix_free(
        weights, operator_representation, states, true_labels,
        backend='numpy', complex_dtype='complex128', chunk_size=64,
        term_chunk_size=8):
    """Classify pure states from matrix-free Hamiltonian expectations."""
    xp, backend_name = resolve_array_backend(backend)
    energies = state_energies_matrix_free(
        weights, operator_representation, states, backend_name, complex_dtype,
        chunk_size, term_chunk_size)
    predictions = xp.sign(energies)
    predictions = xp.where(predictions == 0, 1, predictions)
    labels = xp.asarray(true_labels)
    return float(to_numpy(xp.mean(predictions == labels) * 100, backend_name))

def chebyshev_logloss_coefficients(y, T, spectral_bound, degree):
    """Chebyshev coefficients for T*log(1+exp(-y*x/T)) on [-B,B]."""
    if degree < 1 or spectral_bound <= 0:
        raise ValueError('degree and spectral_bound must be positive')
    sample_count = max(4 * (degree + 1), 256)
    theta = np.pi * (np.arange(sample_count) + 0.5) / sample_count
    nodes = np.cos(theta)
    values = T * np.logaddexp(0, -y * spectral_bound * nodes / T)
    coefficients = (2.0 / sample_count) * (
        np.cos(np.outer(np.arange(degree + 1), theta)) @ values)
    coefficients[0] *= 0.5
    return coefficients

def select_adaptive_chebyshev_degree(
        T, spectral_bound, tolerance=1e-6, min_degree=16,
        max_degree=128, tail_window=8):
    """Choose the smallest degree with a bounded Chebyshev coefficient tail.

    Since |T_k(x)| <= 1 on [-1,1], the sum of omitted coefficient
    magnitudes bounds the sampled polynomial tail uniformly. Both labels are
    checked and the more demanding degree is returned.
    """
    if not (0 < tolerance < 1):
        raise ValueError('Chebyshev tolerance must lie between 0 and 1')
    if not (1 <= min_degree <= max_degree):
        raise ValueError('Require 1 <= min_degree <= max_degree')
    selected = min_degree
    for label in (+1, -1):
        coefficients = chebyshev_logloss_coefficients(
            label, T, spectral_bound, max_degree)
        scale = max(1.0, np.max(np.abs(coefficients)))
        threshold = tolerance * scale
        tail_sums = np.cumsum(np.abs(coefficients[::-1]))[::-1]
        label_degree = max_degree
        # Require both a small omitted tail and a settled final window.
        tail_is_resolved = np.max(np.abs(
            coefficients[-min(tail_window, len(coefficients)):])) <= threshold
        if tail_is_resolved:
            for candidate in range(min_degree, max_degree):
                if tail_sums[candidate + 1] <= threshold:
                    label_degree = candidate
                    break
        selected = max(selected, label_degree)
    return int(selected)

def compute_loss_and_grads_chebyshev(
        weights, state_vectors, ys, operator_representation, T, degree=64,
        chunk_size=32, spectral_bound=None, bound_margin=1.05,
        backend='numpy', complex_dtype='complex128', term_chunk_size=8):
    """Matrix-free Chebyshev loss and reverse-recurrence gradient.

    This evaluates every training state in bounded-memory chunks. The only
    approximation is truncating the Chebyshev series; no dense Hamiltonian,
    eigensystem, density matrix, or label aggregate is constructed.
    """
    xp, backend_name = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    states = xp.asarray(state_vectors, dtype=complex_type)
    labels = xp.asarray(ys)
    numeric_weights = xp.asarray(weights, dtype=real_type)
    num_states = len(states)
    if num_states == 0:
        raise ValueError('state_vectors must not be empty')
    if spectral_bound is None:
        weight_norm = float(to_numpy(xp.sum(xp.abs(numeric_weights)), backend_name))
        spectral_bound = max(bound_margin * weight_norm, 1e-6)
    spectral_bound = float(spectral_bound)
    coefficients = {
        label: xp.asarray(chebyshev_logloss_coefficients(
            label, T, spectral_bound, degree), dtype=real_type)
        for label in (+1, -1)
    }
    loss = xp.asarray(0, dtype=real_type)
    grads = xp.zeros(len(numeric_weights), dtype=real_type)

    def apply_scaled_H(block):
        return apply_pauli_hamiltonian(
            numeric_weights, operator_representation, block, backend_name,
            complex_type, term_chunk_size) / spectral_bound

    def accumulate_parameter_grads(adjoint, forward, factor):
        scale = factor / (spectral_bound * num_states)
        if is_fused_pauli_kernel(operator_representation):
            grads[:] += scale * fused_pauli_bilinears(
                adjoint, forward, operator_representation, backend_name,
                complex_type, term_chunk_size)
        else:
            for j, op in enumerate(operator_representation):
                grads[j] += scale * xp.real(xp.vdot(adjoint, op @ forward))

    for start in range(0, num_states, chunk_size):
        stop = min(start + chunk_size, num_states)
        chunk_states = states[start:stop]
        chunk_labels = labels[start:stop]
        for label in (+1, -1):
            X = chunk_states[chunk_labels == label].T
            if X.shape[1] == 0:
                continue
            coeffs = coefficients[label]
            forward_vectors = [X]
            forward_vectors.append(apply_scaled_H(X))
            for order in range(1, degree):
                forward_vectors.append(
                    2 * apply_scaled_H(forward_vectors[order])
                    - forward_vectors[order - 1])
            for order, vector in enumerate(forward_vectors):
                loss += (coeffs[order] * xp.real(xp.vdot(X, vector))
                         / num_states)

            # Reverse the Chebyshev recurrence without differentiating a
            # dense matrix. lambda_k = c_k X + 2 A lambda_(k+1)-lambda_(k+2).
            adjoint = coeffs[degree] * X
            adjoint_plus_two = xp.zeros_like(X)
            for order in range(degree, 1, -1):
                accumulate_parameter_grads(
                    adjoint, forward_vectors[order - 1], factor=2)
                next_adjoint = (coeffs[order - 1] * X
                                + 2 * apply_scaled_H(adjoint)
                                - adjoint_plus_two)
                adjoint_plus_two, adjoint = adjoint, next_adjoint
            accumulate_parameter_grads(adjoint, forward_vectors[0], factor=1)

    return (float(to_numpy(loss, backend_name)),
            np.asarray(to_numpy(grads, backend_name), dtype=real_type))

def compute_loss_and_grads_aggregated_symbolic(
        weights, label_aggregates, pauli_ops, T, n, sparse_paulis=None,
        backend='numpy', complex_dtype='complex128'):
    """Exact aggregate loss/gradient on an optional CUDA/CuPy backend."""
    xp, backend_name = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    if sparse_paulis is None:
        cpu_sparse = precompute_sparse_paulis(pauli_ops, n)
        sparse_paulis = convert_sparse_paulis_backend(
            cpu_sparse, backend_name, complex_type)
    R_plus, R_minus = (
        xp.asarray(rho, dtype=complex_type) for rho in label_aggregates)
    H_matrix = build_hamiltonian_matrix_backend(
        weights, sparse_paulis, backend_name, complex_type)
    eigvals, eigvecs = xp.linalg.eigh(H_matrix)

    R_plus_tilde = eigvecs.T.conj() @ R_plus @ eigvecs
    R_minus_tilde = eigvecs.T.conj() @ R_minus @ eigvecs
    diag_loss_plus = T * xp.logaddexp(0, -eigvals / T)
    diag_loss_minus = T * xp.logaddexp(0, eigvals / T)
    loss = xp.real(
        xp.dot(diag_loss_plus, xp.diag(R_plus_tilde))
        + xp.dot(diag_loss_minus, xp.diag(R_minus_tilde)))

    F_plus = compute_fdd_matrix_backend(eigvals, y=+1, T=T, backend=backend_name)
    F_minus = compute_fdd_matrix_backend(eigvals, y=-1, T=T, backend=backend_name)
    derivative_eigenbasis = (F_plus * R_plus_tilde.T
                             + F_minus * R_minus_tilde.T)
    derivative_matrix = (eigvecs.conj() @ derivative_eigenbasis
                         @ eigvecs.T)
    grads = xp.stack([
        xp.real(op.multiply(derivative_matrix).sum())
        for op in sparse_paulis
    ]).astype(real_type, copy=False)
    return (float(to_numpy(loss, backend_name)),
            np.asarray(to_numpy(grads, backend_name), dtype=real_type))

def compute_loss_and_grads_fcim_diagonal(
        weights, label_basis_probabilities, feature_matrix, T,
        backend='numpy', complex_dtype='complex128'):
    """Exact diagonal FCIM loss/gradient on NumPy or CuPy."""
    xp, backend_name = resolve_array_backend(backend)
    _, real_type = numeric_dtypes(complex_dtype)
    p_plus, p_minus = (
        xp.asarray(p, dtype=real_type) for p in label_basis_probabilities)
    features = xp.asarray(feature_matrix, dtype=real_type)
    numeric_weights = xp.asarray(weights, dtype=real_type)
    energies = features @ numeric_weights
    loss_plus = T * xp.logaddexp(0, -energies / T)
    loss_minus = T * xp.logaddexp(0, energies / T)
    loss = xp.dot(p_plus, loss_plus) + xp.dot(p_minus, loss_minus)

    derivative_plus = -xp.exp(-xp.logaddexp(0, energies / T))
    derivative_minus = xp.exp(-xp.logaddexp(0, -energies / T))
    weighted_derivative = (p_plus * derivative_plus
                           + p_minus * derivative_minus)
    grads = features.T @ weighted_derivative
    return (float(to_numpy(xp.real(loss), backend_name)),
            np.asarray(to_numpy(xp.real(grads), backend_name), dtype=real_type))

def calculate_accuracy_backend(
        H_model, states, true_labels, backend='numpy',
        complex_dtype='complex128'):
    """Quantum-model accuracy without transferring GPU matrices to CPU."""
    xp, backend_name = resolve_array_backend(backend)
    complex_type, _ = numeric_dtypes(complex_dtype)
    numeric_states = xp.asarray(states, dtype=complex_type)
    if numeric_states.ndim == 2:
        energies = xp.real(xp.einsum(
            'bi,ij,bj->b', numeric_states.conj(), H_model, numeric_states,
            optimize=True))
    elif numeric_states.ndim == 3:
        energies = xp.real(xp.einsum(
            'ij,bji->b', H_model, numeric_states, optimize=True))
    else:
        raise ValueError('states must have shape (M, d) or (M, d, d)')
    predictions = xp.sign(energies)
    predictions = xp.where(predictions == 0, 1, predictions)
    labels = xp.asarray(true_labels)
    return float(to_numpy(xp.mean(predictions == labels) * 100, backend_name))

def calculate_fcim_accuracy(
        weights, feature_matrix, states, true_labels, backend='numpy',
        complex_dtype='complex128'):
    """Classify vectors/density matrices on NumPy or CuPy."""
    xp, backend_name = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    numeric_states = xp.asarray(states, dtype=complex_type)
    features = xp.asarray(feature_matrix, dtype=real_type)
    basis_energies = features @ xp.asarray(weights, dtype=real_type)
    if numeric_states.ndim == 2:
        energies = xp.abs(numeric_states)**2 @ basis_energies
    elif numeric_states.ndim == 3:
        probabilities = xp.real(xp.diagonal(
            numeric_states, axis1=1, axis2=2))
        energies = probabilities @ basis_energies
    else:
        raise ValueError('states must have shape (M, d) or (M, d, d)')
    predictions = xp.sign(xp.real(energies))
    predictions = xp.where(predictions == 0, 1, predictions)
    labels = xp.asarray(true_labels)
    return float(to_numpy(xp.mean(predictions == labels) * 100, backend_name))

def compute_loss_and_grads_symbolic(
        weights, training_states, ys, pauli_ops, T, n, sparse_paulis=None):
    """Compute exact loss/gradients from symbolic terms plus a sparse cache.

    The full weighted H is dense only for ``np.linalg.eigh``. Each derivative
    term remains CSR when acting on the eigenvectors, avoiding a dense Pauli
    basis in memory and reducing the cost of P_j @ V.
    """
    if sparse_paulis is None:
        sparse_paulis = precompute_sparse_paulis(pauli_ops, n)

    H_matrix = build_hamiltonian_matrix_from_symbolic(
        weights, pauli_ops, n, sparse_paulis=sparse_paulis)
    eigvals, eigvecs = np.linalg.eigh(H_matrix)

    # logaddexp is stable for large |eigenvalue / T|.
    diag_loss_plus = T * np.logaddexp(0, -eigvals / T)
    diag_loss_minus = T * np.logaddexp(0, eigvals / T)

    rhos_tilde = np.array([eigvecs.T.conj() @ rho @ eigvecs
                            for rho in training_states])
    rhos_diag = np.real(np.diagonal(rhos_tilde, axis1=1, axis2=2))
    loss_diagonals = np.where(ys[:, None] > 0, diag_loss_plus, diag_loss_minus)
    loss = np.mean(np.sum(loss_diagonals * rhos_diag, axis=1))

    F_plus = compute_fdd_matrix(eigvals, y=+1, T=T)
    F_minus = compute_fdd_matrix(eigvals, y=-1, T=T)
    grads = np.zeros(len(weights))

    for j, H_j_sparse in enumerate(sparse_paulis):
        # CSR Pauli action costs O(4^n); dense P_j @ V would cost O(8^n).
        H_j_tilde = eigvecs.T.conj() @ (H_j_sparse @ eigvecs)
        grad_j = 0.0
        for i, y_i in enumerate(ys):
            F = F_plus if y_i > 0 else F_minus
            grad_j += np.real(np.einsum(
                'ij,ij,ij->', F, H_j_tilde, rhos_tilde[i].T))
        grads[j] = grad_j / len(training_states)

    return loss, grads


In [5]:
# optimize: training protocol of Paper Sec. VI.C applied to logistic-loss /
#   binary classification (Sec. VI.D.2, Fig. 8 + Table II). Trains the quantum
#   Heisenberg model and the classical FCIM in parallel on the same labeled data.
#   
#   OPTIMIZED VERSION: Vectorizes gradient computation across entire training set
#   at once (rather than looping over dfj() for each parameter), providing ~3-5x speedup.
def optimize(n=3, epochs=2000, use_fast_grad=True):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    val_states = make_validation_set(n, num_states=500)
    T = 2.0   # temperature T in the logistic loss Eq. (56)
    
    # Data generation: random target Hamiltonian and labels
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q))
    ys = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in training_states])
    ys[ys == 0] = 1

    ys_val = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in val_states])
    ys_val[ys_val == 0] = 1

    # Initialize parameters
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    eta = 0.1

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits (Fast Gradients={use_fast_grad}) ---")
    print(f"{'Epoch':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")    
    print("-" * 60)
    
    for epoch in range(epochs):
        # --- Quantum Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            # Use vectorized gradient computation (~3-5x faster)
            l_q, grad_q = compute_loss_and_grads_vectorized(est_q, training_states, ys, pauli_q, T)
        else:
            # Original implementation: loop over dfj() for each parameter
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            eval_q, evec_q = np.linalg.eigh(H_q)
            l_q = 0
            for i in range(N_states):
                yi = ys[i]
                m_loss_q = evec_q @ np.diag(T * np.log(1 + np.exp(-yi * eval_q / T))) @ evec_q.T.conj()
                l_q += np.real(np.trace(m_loss_q @ training_states[i]))
            l_q /= N_states
            
            grad_q = np.zeros(len(pauli_q))
            for j in range(len(pauli_q)):
                g_j = sum(dfj(ys[i], training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
                grad_q[j] = g_j / N_states
        
        history_q.append(l_q)
        est_q -= eta * grad_q

        # --- Classical Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            l_c, grad_c = compute_loss_and_grads_vectorized(est_c, training_states, ys, pauli_c, T)
        else:
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            eval_c, evec_c = np.linalg.eigh(H_c)
            l_c = 0.0
            for i in range(N_states):
                yi = ys[i]
                m_loss_c = evec_c @ np.diag(T * np.log(1 + np.exp(-yi * eval_c / T))) @ evec_c.T.conj()
                l_c += np.real(np.trace(m_loss_c @ training_states[i]))
            l_c /= N_states
            
            grad_c = np.zeros(len(pauli_c))
            for j in range(len(pauli_c)):
                g_j = sum(dfj(ys[i], training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
                grad_c[j] = g_j / N_states
        
        history_c.append(l_c)
        est_c -= eta * grad_c

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
            "Quantum_Accuracy_Pct": None,
            "Classical_Accuracy_Pct": None
        }

        if epoch % 20 == 0:
            # Compute validation accuracy
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            q_acc = calculate_accuracy(H_q, val_states, ys_val)
            c_acc = calculate_accuracy(H_c, val_states, ys_val)

            epoch_data["Quantum_Accuracy_Pct"] = q_acc
            epoch_data["Classical_Accuracy_Pct"] = c_acc
            print(f"{epoch:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    final_H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
    final_H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_cl_heisenberg.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {calculate_accuracy(final_H_q, val_states, ys_val):.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_accuracy(final_H_c, val_states, ys_val):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [8]:
# optimize_phase2: exact symbolic/aggregate implementation with GD or L-BFGS
def optimize_phase2(
        n=3, epochs=2000, optimizer='gd', learning_rate=0.1,
        adam_learning_rate=0.01, adam_batch_size=64, adam_beta1=0.9,
        adam_beta2=0.999, adam_epsilon=1e-8, adam_ema_beta=0.9,
        adam_seed=None,
        early_stopping_patience=50, early_stopping_tol=1e-8,
        validation_frequency=20, lbfgs_ftol=1e-12, lbfgs_gtol=1e-6,
        backend='numpy', complex_dtype='complex128', quantum_method='auto',
        chebyshev_degree=64, chebyshev_chunk_size=32,
        chebyshev_spectral_bound=None, chebyshev_bound_margin=1.05,
        adaptive_chebyshev=True, chebyshev_tolerance=1e-6,
        chebyshev_min_degree=16, chebyshev_max_degree=128,
        pauli_kernel='auto', pauli_term_chunk_size=8,
        num_training_states=1000, num_validation_states=500):
    """Train the quantum and FCIM models with analytic gradients.

    ``optimizer`` is ``'gd'``, ``'adam'``, or ``'lbfgs'``. Adam resamples
    quantum-state mini-batches while retaining the inexpensive full FCIM loss.
    ``epochs`` is the maximum number of GD epochs or L-BFGS iterations.
    Patience-based early stopping monitors the sum of quantum and classical
    losses; use ``early_stopping_patience=None`` to disable it.
    ``backend='cupy'`` enables CUDA when CuPy is installed; ``'auto'`` uses
    CUDA when available and otherwise falls back to NumPy.
    ``quantum_method='auto'`` selects exact eigendecomposition through n=10
    and the matrix-free Chebyshev approximation for n>10.
    ``pauli_kernel='auto'`` uses CSR on CPU and fused bitwise actions on GPU.
    """
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    xp, backend_name = resolve_array_backend(backend)
    complex_type, real_type = numeric_dtypes(complex_dtype)
    if complex_type == np.complex64:
        # Avoid asking a single-precision objective for double precision.
        lbfgs_ftol = max(lbfgs_ftol, 1e-7)
        lbfgs_gtol = max(lbfgs_gtol, 1e-5)
    method_name = quantum_method.lower().replace('-', '').replace('_', '')
    if method_name == 'auto':
        method_name = 'chebyshev' if n > 10 else 'eigh'
    if method_name in {'exact', 'dense'}:
        method_name = 'eigh'
    if method_name not in {'eigh', 'chebyshev'}:
        raise ValueError("quantum_method must be 'auto', 'eigh', or 'chebyshev'")
    kernel_name = pauli_kernel.lower().replace('-', '').replace('_', '')
    if kernel_name == 'auto':
        kernel_name = 'fused' if backend_name == 'cupy' else 'csr'
    if kernel_name not in {'csr', 'fused'}:
        raise ValueError("pauli_kernel must be 'auto', 'csr', or 'fused'")
    if method_name == 'eigh':
        kernel_name = 'csr'
    
    # Exact mode uses CSR; Chebyshev uses fused bitwise permutations/phases.
    pauli_q_sym = generate_paulis_symbolic(n, model="quantum")
    if method_name == 'chebyshev' and kernel_name == 'fused':
        quantum_representation = build_fused_pauli_kernel(
            n, model='quantum', backend=backend_name,
            complex_dtype=complex_type)
    else:
        sparse_q_cpu = precompute_sparse_paulis_direct(
            n, model='quantum', complex_dtype=complex_type)
        quantum_representation = convert_sparse_paulis_backend(
            sparse_q_cpu, backend_name, complex_type)
    # Classical FCIM: exact computational-basis feature representation.
    fcim_features_cpu = build_fcim_feature_matrix(n)

    # Preserve pure states as vectors; never allocate M dense density matrices.
    training_states = make_state_vectors(
        n, num_states=num_training_states, complex_dtype=complex_type)
    val_states = make_state_vectors(
        n, num_states=num_validation_states, complex_dtype=complex_type)
    training_states_backend = xp.asarray(training_states, dtype=complex_type)
    val_states_backend = xp.asarray(val_states, dtype=complex_type)
    T = 2.0
    
    # Data generation: target labels are also evaluated matrix-free.
    target_p = (np.random.random(len(pauli_q_sym)) - 0.5) * 4
    target_training_energies = state_energies_matrix_free(
        target_p, quantum_representation, training_states_backend,
        backend_name, complex_type, chebyshev_chunk_size,
        pauli_term_chunk_size)
    target_validation_energies = state_energies_matrix_free(
        target_p, quantum_representation, val_states_backend, backend_name,
        complex_type, chebyshev_chunk_size, pauli_term_chunk_size)
    ys = np.sign(to_numpy(target_training_energies, backend_name))
    ys[ys == 0] = 1

    ys_val = np.sign(to_numpy(target_validation_energies, backend_name))
    ys_val[ys_val == 0] = 1

    # FCIM always needs only basis probabilities. The exact quantum path uses
    # two dense aggregates; Chebyshev retains vectors and creates neither.
    label_basis_probabilities = aggregate_basis_probabilities_by_label(
        training_states, ys)
    if method_name == 'eigh':
        label_aggregates = aggregate_states_by_label(training_states, ys)
        label_aggregates = tuple(
            xp.asarray(rho, dtype=complex_type) for rho in label_aggregates)
        del training_states_backend
    else:
        label_aggregates = None
    label_basis_probabilities = tuple(
        xp.asarray(p, dtype=real_type) for p in label_basis_probabilities)
    fcim_features = xp.asarray(fcim_features_cpu, dtype=real_type)
    del training_states, val_states

    # Initialize parameters
    est_q = (np.random.random(len(pauli_q_sym)) - 0.5)
    est_c = (np.random.random(fcim_features_cpu.shape[1]) - 0.5)
    
    eta = learning_rate

    history_q = []
    history_c = []

    optimizer_name = optimizer.lower().replace('-', '').replace('_', '')
    if optimizer_name == 'lbfgsb':
        optimizer_name = 'lbfgs'
    if optimizer_name not in {'gd', 'adam', 'lbfgs'}:
        raise ValueError("optimizer must be 'gd', 'adam', or 'lbfgs'")
    if early_stopping_tol < 0:
        raise ValueError('early_stopping_tol must be non-negative')
    if adam_batch_size < 1:
        raise ValueError('adam_batch_size must be positive')
    if not (0 <= adam_beta1 < 1 and 0 <= adam_beta2 < 1
            and 0 <= adam_ema_beta < 1):
        raise ValueError('Adam beta values must lie in [0, 1)')
    adaptive_degree_enabled = (adaptive_chebyshev
                               and optimizer_name != 'lbfgs')
    if adaptive_chebyshev and optimizer_name == 'lbfgs' and method_name == 'chebyshev':
        print('Adaptive Chebyshev degree disabled for smooth L-BFGS line searches.')

    print(f"--- Phase 2: {n} Qubits, {optimizer.upper()}, {method_name}/{kernel_name}, {backend_name}/{np.dtype(complex_type).name} ---")
    print(f"{'Step':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")
    print("-" * 60)
    chebyshev_state = {'degree': None, 'spectral_bound': None}

    def evaluate_models(q_weights, c_weights, batch_indices=None):
        if method_name == 'eigh':
            l_q, grad_q = compute_loss_and_grads_aggregated_symbolic(
                q_weights, label_aggregates, pauli_q_sym, T, n,
                quantum_representation,
                backend=backend_name, complex_dtype=complex_type)
        else:
            if batch_indices is None:
                q_states = training_states_backend
                q_labels = ys
            else:
                device_indices = xp.asarray(batch_indices)
                q_states = training_states_backend[device_indices]
                q_labels = ys[batch_indices]
            active_bound = chebyshev_spectral_bound
            if active_bound is None:
                active_bound = max(
                    chebyshev_bound_margin * np.sum(np.abs(q_weights)), 1e-6)
            active_degree = chebyshev_degree
            if adaptive_degree_enabled:
                active_degree = select_adaptive_chebyshev_degree(
                    T, active_bound, tolerance=chebyshev_tolerance,
                    min_degree=chebyshev_min_degree,
                    max_degree=chebyshev_max_degree)
            chebyshev_state['degree'] = active_degree
            chebyshev_state['spectral_bound'] = active_bound
            l_q, grad_q = compute_loss_and_grads_chebyshev(
                q_weights, q_states, q_labels,
                quantum_representation, T,
                degree=active_degree, chunk_size=chebyshev_chunk_size,
                spectral_bound=active_bound,
                bound_margin=chebyshev_bound_margin, backend=backend_name,
                complex_dtype=complex_type,
                term_chunk_size=pauli_term_chunk_size)
        l_c, grad_c = compute_loss_and_grads_fcim_diagonal(
            c_weights, label_basis_probabilities, fcim_features, T,
            backend=backend_name, complex_dtype=complex_type)
        return l_q, grad_q, l_c, grad_c

    def record_iteration(
            step, q_weights, c_weights, l_q, l_c, batch_size=None):
        history_q.append(float(l_q))
        history_c.append(float(l_c))
        step_data = {
            'Epoch': step,
            'Optimizer': optimizer_name,
            'Backend': backend_name,
            'Complex_Dtype': np.dtype(complex_type).name,
            'Quantum_Method': method_name,
            'Pauli_Kernel': kernel_name,
            'Chebyshev_Degree': chebyshev_state['degree'],
            'Chebyshev_Spectral_Bound': chebyshev_state['spectral_bound'],
            'Batch_Size': batch_size,
            'Quantum_Loss': float(l_q),
            'Classical_Loss': float(l_c),
            'Quantum_Accuracy_Pct': None,
            'Classical_Accuracy_Pct': None
        }
        if validation_frequency and step % validation_frequency == 0:
            q_acc = calculate_accuracy_matrix_free(
                q_weights, quantum_representation, val_states_backend, ys_val,
                backend_name, complex_type, chebyshev_chunk_size,
                pauli_term_chunk_size)
            c_acc = calculate_fcim_accuracy(
                c_weights, fcim_features, val_states_backend, ys_val,
                backend_name, complex_type)
            step_data['Quantum_Accuracy_Pct'] = q_acc
            step_data['Classical_Accuracy_Pct'] = c_acc
            print(f"{step:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(step_data)

    if optimizer_name == 'gd':
        best_total = np.inf
        best_weights = (est_q.copy(), est_c.copy())
        stale_steps = 0
        for step in range(epochs):
            l_q, grad_q, l_c, grad_c = evaluate_models(est_q, est_c)
            record_iteration(step, est_q, est_c, l_q, l_c)
            total_loss = l_q + l_c
            if total_loss < best_total - early_stopping_tol:
                best_total = total_loss
                best_weights = (est_q.copy(), est_c.copy())
                stale_steps = 0
            else:
                stale_steps += 1
            if (early_stopping_patience is not None
                    and stale_steps >= early_stopping_patience):
                est_q, est_c = best_weights
                print(f"Early stopping at step {step}; restored best weights.")
                break
            est_q -= eta * grad_q
            est_c -= eta * grad_c
    elif optimizer_name == 'adam':
        adam_rng = np.random.default_rng(adam_seed)
        m_q, v_q = np.zeros_like(est_q), np.zeros_like(est_q)
        m_c, v_c = np.zeros_like(est_c), np.zeros_like(est_c)
        best_monitor = np.inf
        best_weights = (est_q.copy(), est_c.copy())
        stale_steps = 0
        ema_loss = None
        for step in range(epochs):
            if method_name == 'chebyshev' and adam_batch_size < num_training_states:
                batch_indices = adam_rng.choice(
                    num_training_states, size=adam_batch_size, replace=False)
            else:
                batch_indices = None
            effective_batch_size = (num_training_states if batch_indices is None
                                    else len(batch_indices))
            l_q, grad_q, l_c, grad_c = evaluate_models(
                est_q, est_c, batch_indices=batch_indices)
            record_iteration(
                step, est_q, est_c, l_q, l_c,
                batch_size=effective_batch_size)

            iteration = step + 1
            m_q = adam_beta1 * m_q + (1 - adam_beta1) * grad_q
            v_q = adam_beta2 * v_q + (1 - adam_beta2) * grad_q**2
            m_c = adam_beta1 * m_c + (1 - adam_beta1) * grad_c
            v_c = adam_beta2 * v_c + (1 - adam_beta2) * grad_c**2
            m_q_hat = m_q / (1 - adam_beta1**iteration)
            v_q_hat = v_q / (1 - adam_beta2**iteration)
            m_c_hat = m_c / (1 - adam_beta1**iteration)
            v_c_hat = v_c / (1 - adam_beta2**iteration)

            batch_total = float(l_q + l_c)
            ema_loss = (batch_total if ema_loss is None else
                        adam_ema_beta * ema_loss
                        + (1 - adam_ema_beta) * batch_total)
            if ema_loss < best_monitor - early_stopping_tol:
                best_monitor = ema_loss
                best_weights = (est_q.copy(), est_c.copy())
                stale_steps = 0
            else:
                stale_steps += 1
            if (early_stopping_patience is not None
                    and stale_steps >= early_stopping_patience):
                est_q, est_c = best_weights
                print(f"Adam early stopping at step {step}; restored best EMA weights.")
                break

            est_q -= adam_learning_rate * m_q_hat / (np.sqrt(v_q_hat) + adam_epsilon)
            est_c -= adam_learning_rate * m_c_hat / (np.sqrt(v_c_hat) + adam_epsilon)
    else:
        n_q = len(est_q)
        theta0 = np.concatenate([est_q, est_c])
        cache = {'theta': None}
        stop_state = {
            'step': 0, 'best_total': np.inf, 'best_theta': theta0.copy(),
            'stale': 0, 'stopped_early': False
        }

        def joint_objective(theta):
            if cache['theta'] is not None and np.array_equal(theta, cache['theta']):
                return cache['total'], cache['gradient']
            q_weights, c_weights = theta[:n_q], theta[n_q:]
            l_q, grad_q, l_c, grad_c = evaluate_models(q_weights, c_weights)
            cache.update({
                'theta': theta.copy(), 'total': float(l_q + l_c),
                'gradient': np.concatenate([grad_q, grad_c]),
                'l_q': float(l_q), 'l_c': float(l_c)
            })
            return cache['total'], cache['gradient']

        def lbfgs_callback(theta):
            total_loss, _ = joint_objective(theta)
            step = stop_state['step']
            record_iteration(
                step, theta[:n_q], theta[n_q:], cache['l_q'], cache['l_c'])
            if total_loss < stop_state['best_total'] - early_stopping_tol:
                stop_state['best_total'] = total_loss
                stop_state['best_theta'] = theta.copy()
                stop_state['stale'] = 0
            else:
                stop_state['stale'] += 1
            stop_state['step'] += 1
            if (early_stopping_patience is not None
                    and stop_state['stale'] >= early_stopping_patience):
                stop_state['stopped_early'] = True
                raise StopIteration

        result = minimize(
            joint_objective, theta0, method='L-BFGS-B', jac=True,
            callback=lbfgs_callback,
            options={
                'maxiter': epochs, 'ftol': lbfgs_ftol,
                'gtol': lbfgs_gtol, 'maxls': 20
            })
        final_theta = (stop_state['best_theta']
                       if stop_state['stopped_early'] else result.x)
        est_q, est_c = final_theta[:n_q].copy(), final_theta[n_q:].copy()
        if not history_q:
            l_q, _, l_c, _ = evaluate_models(est_q, est_c)
            record_iteration(0, est_q, est_c, l_q, l_c)
        if stop_state['stopped_early']:
            print(f"Early stopping after {stop_state['step']} L-BFGS iterations; restored best weights.")
        else:
            print(f"L-BFGS: {result.message}")

    print("\n--- Final Results ---")
    final_q_accuracy = calculate_accuracy_matrix_free(
        est_q, quantum_representation, val_states_backend, ys_val,
        backend_name, complex_type, chebyshev_chunk_size,
        pauli_term_chunk_size)
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_phase2.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {final_q_accuracy:.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_fcim_accuracy(est_c, fcim_features, val_states_backend, ys_val, backend_name, complex_type):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# plot: reproduces a panel of Paper Fig. 8 -- logistic loss vs epoch for the
#   quantum Heisenberg model vs the classical FCIM.
def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_{\text{FCIM}}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_{\text{Heis}}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Logistic Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    
    plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    plt.title(f'{n} Qubits, Heisenberg Model', fontsize=24)
    plt.subplots_adjust(left=0.13, right=0.97, bottom=0.18, top=0.90)
    # plt.yscale('log')
    plt.legend(fontsize=24)
    plt.savefig(f"plots/logloss_{n}qubit_heis_fcim.pdf", format="pdf")
    plt.show()

In [6]:
# === COMPREHENSIVE BENCHMARK: Phase 1 + Phase 3 Optimization ===
import time
import tracemalloc

print("="*70)
print("BENCHMARK: Phase 1 + Phase 3 Optimization (Vectorized Gradients)")
print("="*70)

# Test on n=4 (larger than n=3 for better timing accuracy)
n = 4
test_epochs = [5, 10, 20]

print(f"\nSystem: {n} qubits")
print(f"Training set: 1000 states")
print(f"Validation set: 500 states")
print(f"\n{'Epochs':<8} | {'Original (s)':<14} | {'Optimized (s)':<14} | {'Speedup':<8}")
print("-" * 70)

results = []

for epochs in test_epochs:
    # Benchmark original implementation
    tracemalloc.start()
    start = time.perf_counter()
    history_q_orig, history_c_orig = optimize(n, epochs=epochs, use_fast_grad=False)
    time_orig = time.perf_counter() - start
    current_mem, peak_mem_orig = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # Benchmark optimized implementation  
    tracemalloc.start()
    start = time.perf_counter()
    history_q_opt, history_c_opt = optimize(n, epochs=epochs, use_fast_grad=True)
    time_opt = time.perf_counter() - start
    current_mem, peak_mem_opt = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    speedup = time_orig / time_opt
    results.append({
        'epochs': epochs,
        'time_orig': time_orig,
        'time_opt': time_opt,
        'speedup': speedup,
        'mem_orig': peak_mem_orig,
        'mem_opt': peak_mem_opt
    })
    
    print(f"{epochs:<8} | {time_orig:<14.2f} | {time_opt:<14.2f} | {speedup:<8.2f}x")

print("\n" + "="*70)
print("SPEEDUP ANALYSIS & EXTRAPOLATION")
print("="*70)

avg_speedup = np.mean([r['speedup'] for r in results])
print(f"\nAverage speedup (n={n}): {avg_speedup:.2f}x")

# Estimate time for n=6
print(f"\n{'─'*70}")
print(f"EXTRAPOLATION TO FULL TRAINING (n=6, 750 epochs):")
print(f"{'─'*70}")

# Estimate based on per-epoch time
per_epoch_orig = results[-1]['time_orig'] / results[-1]['epochs']
per_epoch_opt = results[-1]['time_opt'] / results[-1]['epochs']

total_time_orig_est = per_epoch_orig * 750 / 60  # Convert to minutes
total_time_opt_est = per_epoch_opt * 750 / 60

time_saved = total_time_orig_est - total_time_opt_est

print(f"\nPer-epoch time (n={n}):")
print(f"  Original:  {per_epoch_orig:.3f}s")
print(f"  Optimized: {per_epoch_opt:.3f}s")
print(f"  Saved per epoch: {per_epoch_orig - per_epoch_opt:.3f}s ({(1-per_epoch_opt/per_epoch_orig)*100:.1f}% faster)")

print(f"\nEstimated full training (n=6, 750 epochs):")
print(f"  Original:  {total_time_orig_est:.1f} minutes ({total_time_orig_est/60:.1f} hours)")
print(f"  Optimized: {total_time_opt_est:.1f} minutes ({total_time_opt_est/60:.1f} hours)")
print(f"  Time saved: {time_saved:.1f} minutes ({time_saved*60:.0f} seconds)")
print(f"  Overall speedup: {avg_speedup:.2f}x")

print(f"\n{'─'*70}")
print(f"NUMERICAL ACCURACY")
print(f"{'─'*70}")

# Check numerical equivalence
loss_diff_q = np.max(np.abs(np.array(history_q_orig) - np.array(history_q_opt)))
loss_diff_c = np.max(np.abs(np.array(history_c_orig) - np.array(history_c_opt)))

print(f"\nLoss difference (should be < 0.1 for numerical agreement):")
print(f"  Quantum model: {loss_diff_q:.2e}")
print(f"  Classical model: {loss_diff_c:.2e}")

if loss_diff_q < 0.1 and loss_diff_c < 0.1:
    print(f"\n✓ PASS: Implementations are numerically equivalent")
else:
    print(f"\n⚠ WARNING: Significant difference detected")

print(f"\n{'─'*70}")
print(f"MEMORY USAGE")
print(f"{'─'*70}")

print(f"\nPeak memory (n={n}, {results[-1]['epochs']} epochs):")
print(f"  Original:  {results[-1]['mem_orig']/1e6:.1f} MB")
print(f"  Optimized: {results[-1]['mem_opt']/1e6:.1f} MB")

print(f"\n" + "="*70)
print(f"SUMMARY: Phase 1 + Phase 3 provides {avg_speedup:.2f}x speedup")
print(f"="*70)

BENCHMARK: Phase 1 + Phase 3 Optimization (Vectorized Gradients)

System: 4 qubits
Training set: 1000 states
Validation set: 500 states

Epochs   | Original (s)   | Optimized (s)  | Speedup 
----------------------------------------------------------------------
--- Running Optimization for 4 Qubits (Fast Gradients=False) ---
Epoch  | Q Loss     | C Loss     | Q Acc (%)  | C Acc (%) 
------------------------------------------------------------
0      | 1.48815    | 1.41855    | 57.00      | 48.80     

--- Final Results ---
Final Quantum Validation Accuracy:   57.40%
Final Classical Validation Accuracy: 49.20%

Target Params:  [ 1.36252  1.14719 -0.86469 -1.6688   0.81178  0.04944  1.28256  1.07616
  1.43142 -0.29369 -1.10294  1.8309  -1.65838  1.34454  1.44997 -1.13263
  0.58176 -1.90599  0.6323  -1.67701 -0.47523]
Estimated Q:    [-0.289    0.41644  0.05179  0.34804 -0.40938 -0.26745  0.37297  0.01703
 -0.20663 -0.44198 -0.46223  0.36717  0.08601  0.28929  0.39267  0.44348
 -0.22016 -

In [ ]:
# === PHASE 2 BENCHMARK: Compare Phase 1+3 vs Phase 2 (Symbolic Operators) ===
import time

print("\n" + "="*80)
print("BENCHMARK: Phase 1+3 vs Phase 2 (Symbolic Pauli Operators)")
print("="*80)

n = 4
test_epochs = [5]  # Start with 5 epochs for quick comparison

print(f"\nSystem: {n} qubits")
print(f"Training set: 1000 states")
print(f"Validation set: 500 states")
print(f"\n{'Epochs':<8} | {'Phase 1+3 (s)':<16} | {'Phase 2 (s)':<16} | {'Speedup':<10}")
print("-" * 80)

phase2_results = []
benchmark_seed = 20260622

for epochs in test_epochs:
    # Phase 1+3 Benchmark
    print(f"\n[Phase 1+3 - {epochs} epochs]")
    np.random.seed(benchmark_seed)
    start = time.perf_counter()
    history_q_p1, history_c_p1 = optimize(n, epochs=epochs, use_fast_grad=True)
    time_p1 = time.perf_counter() - start
    
    # Phase 2 Benchmark
    print(f"\n[Phase 2 - {epochs} epochs]")
    np.random.seed(benchmark_seed)  # identical data and initialization
    start = time.perf_counter()
    history_q_p2, history_c_p2 = optimize_phase2(n, epochs=epochs)
    time_p2 = time.perf_counter() - start
    
    speedup_p2_vs_p1 = time_p1 / time_p2 if time_p2 > 0 else 0
    
    phase2_results.append({
        'epochs': epochs,
        'time_p1': time_p1,
        'time_p2': time_p2,
        'speedup': speedup_p2_vs_p1
    })
    
    print(f"\n{epochs:<8} | {time_p1:<16.2f} | {time_p2:<16.2f} | {speedup_p2_vs_p1:<10.2f}x")
    
    # Numerical accuracy check
    loss_diff_q = np.max(np.abs(np.array(history_q_p1) - np.array(history_q_p2)))
    loss_diff_c = np.max(np.abs(np.array(history_c_p1) - np.array(history_c_p2)))
    
    print(f"\nNumerical accuracy:")
    print(f"  Quantum model loss diff:   {loss_diff_q:.2e}")
    print(f"  Classical model loss diff: {loss_diff_c:.2e}")
    
    if loss_diff_q < 0.1 and loss_diff_c < 0.1:
        print(f"  ✓ PASS: Implementations are numerically equivalent")
    else:
        print(f"  ⚠ WARNING: Significant difference detected")

print(f"\n" + "="*80)
print("PHASE 2 INTEGRATION RESULTS")
print("="*80)

if phase2_results:
    avg_speedup_p2 = np.mean([r['speedup'] for r in phase2_results])
    print(f"\nPhase 2 vs Phase 1+3 speedup: {avg_speedup_p2:.2f}x")
    print(f"\nNote: both paths use the same dense exact eigendecomposition.")
    print(f"      The n=6 cell below measures the demonstrated representation")
    print(f"      advantage separately from end-to-end training time.")


In [ ]:
# === N=6 SYMBOLIC REPRESENTATION REGRESSION + SCALING BENCHMARK ===

n_test = 6
rng = np.random.default_rng(20260622)
weights = rng.uniform(-0.5, 0.5, 6 * n_test - 3)

dense_terms = generate_paulis(n_test, model='quantum')
symbolic_terms = generate_paulis_symbolic(n_test, model='quantum')
pennylane_sparse_terms = precompute_sparse_paulis(symbolic_terms, n_test)
sparse_terms = precompute_sparse_paulis_direct(n_test, model='quantum')
fused_terms = build_fused_pauli_kernel(n_test, model='quantum')
direct_sparse_error = max(np.max(np.abs((a - b).toarray()))
                          for a, b in zip(pennylane_sparse_terms, sparse_terms))

assert all(isinstance(op, qml.operation.Operator) for op in symbolic_terms)
assert isinstance(build_hamiltonian_symbolic(weights, symbolic_terms), qml.operation.Operator)

dense_bytes = sum(op.nbytes for op in dense_terms)
sparse_bytes = sparse_storage_bytes(sparse_terms)
memory_reduction = dense_bytes / sparse_bytes
dense_entries = sum(op.size for op in dense_terms)
sparse_nonzeros = sum(op.nnz for op in sparse_terms)
entry_reduction = dense_entries / sparse_nonzeros

H_dense = sum((w * op for w, op in zip(weights, dense_terms)),
              start=np.zeros_like(dense_terms[0]))
H_symbolic = build_hamiltonian_matrix_from_symbolic(
    weights, symbolic_terms, n_test, sparse_paulis=sparse_terms)
hamiltonian_error = np.max(np.abs(H_dense - H_symbolic))

# Exact loss/gradient regression on n=6 pure-state density matrices.
num_benchmark_states = 64
vectors = (rng.normal(size=(num_benchmark_states, 2**n_test))
           + 1j * rng.normal(size=(num_benchmark_states, 2**n_test)))
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
benchmark_states = np.einsum('bi,bj->bij', vectors, vectors.conj())
benchmark_labels = np.sign(np.real(np.einsum(
    'bi,ij,bj->b', vectors.conj(), H_dense, vectors)))
benchmark_labels[benchmark_labels == 0] = 1
fused_action_error = np.max(np.abs(
    apply_pauli_hamiltonian(weights, sparse_terms, vectors.T)
    - apply_pauli_hamiltonian(weights, fused_terms, vectors.T)))

dense_result = compute_loss_and_grads_vectorized(
    weights, benchmark_states, benchmark_labels, dense_terms, T=2.0)
symbolic_result = compute_loss_and_grads_symbolic(
    weights, benchmark_states, benchmark_labels, symbolic_terms, T=2.0,
    n=n_test, sparse_paulis=sparse_terms)
loss_error = abs(dense_result[0] - symbolic_result[0])
gradient_error = np.max(np.abs(dense_result[1] - symbolic_result[1]))

# Exact label-aggregate quantum regression.
label_aggregates = aggregate_states_by_label(vectors, benchmark_labels)
aggregated_result = compute_loss_and_grads_aggregated_symbolic(
    weights, label_aggregates, symbolic_terms, T=2.0, n=n_test,
    sparse_paulis=sparse_terms)
aggregate_loss_error = abs(symbolic_result[0] - aggregated_result[0])
aggregate_gradient_error = np.max(
    np.abs(symbolic_result[1] - aggregated_result[1]))
aggregate_memory_reduction = (benchmark_states.nbytes
                              / sum(rho.nbytes for rho in label_aggregates))

# Matrix-free Chebyshev regression against the exact quantum path.
chebyshev_bound = 1.05 * np.sum(np.abs(weights))
adaptive_test_degree = select_adaptive_chebyshev_degree(
    T=2.0, spectral_bound=chebyshev_bound, tolerance=1e-8,
    min_degree=8, max_degree=128)
chebyshev_result = compute_loss_and_grads_chebyshev(
    weights, vectors, benchmark_labels, fused_terms, T=2.0,
    degree=adaptive_test_degree,
    chunk_size=16, spectral_bound=chebyshev_bound)
chebyshev_loss_error = abs(symbolic_result[0] - chebyshev_result[0])
chebyshev_gradient_error = np.max(
    np.abs(symbolic_result[1] - chebyshev_result[1]))

# Exact diagonal-FCIM regression against the generic eigensolver path.
classical_terms = generate_paulis_symbolic(n_test, model='classical')
classical_sparse_terms = precompute_sparse_paulis(classical_terms, n_test)
classical_weights = rng.uniform(-0.5, 0.5, len(classical_terms))
classical_generic_result = compute_loss_and_grads_symbolic(
    classical_weights, benchmark_states, benchmark_labels, classical_terms,
    T=2.0, n=n_test, sparse_paulis=classical_sparse_terms)
fcim_features = build_fcim_feature_matrix(n_test)
label_basis_probabilities = aggregate_basis_probabilities_by_label(
    vectors, benchmark_labels)
classical_diagonal_result = compute_loss_and_grads_fcim_diagonal(
    classical_weights, label_basis_probabilities, fcim_features, T=2.0)
fcim_loss_error = abs(
    classical_generic_result[0] - classical_diagonal_result[0])
fcim_gradient_error = np.max(np.abs(
    classical_generic_result[1] - classical_diagonal_result[1]))

assert hamiltonian_error < 1e-12
assert direct_sparse_error < 1e-12
assert fused_action_error < 1e-12
assert loss_error < 1e-12
assert gradient_error < 1e-12
assert aggregate_loss_error < 1e-12
assert aggregate_gradient_error < 1e-12
assert chebyshev_loss_error < 1e-8
assert chebyshev_gradient_error < 1e-8
assert fcim_loss_error < 1e-12
assert fcim_gradient_error < 1e-12
assert memory_reduction > 10
assert entry_reduction == 2**n_test

print(f'n={n_test}: {len(symbolic_terms)} genuine PennyLane operators')
print(f'Dense term payload:  {dense_bytes / 2**20:.3f} MiB')
print(f'Sparse term payload: {sparse_bytes / 2**10:.3f} KiB')
print(f'Term-storage reduction: {memory_reduction:.2f}x')
print(f'Stored numeric entries: {dense_entries:,} dense vs {sparse_nonzeros:,} sparse')
print(f'Entry-count reduction: {entry_reduction:.0f}x')
print(f'Hamiltonian max error: {hamiltonian_error:.3e}')
print(f'Direct CSR max error: {direct_sparse_error:.3e}')
print(f'Fused action max error: {fused_action_error:.3e}')
print(f'Loss error: {loss_error:.3e}')
print(f'Gradient max error: {gradient_error:.3e}')
print(f'Aggregate loss/gradient errors: {aggregate_loss_error:.3e} / {aggregate_gradient_error:.3e}')
print(f'Aggregate-state memory reduction ({num_benchmark_states} states): {aggregate_memory_reduction:.2f}x')
print(f'Chebyshev loss/gradient errors: {chebyshev_loss_error:.3e} / {chebyshev_gradient_error:.3e}')
print(f'Adaptive Chebyshev degree (1e-8 tolerance): {adaptive_test_degree}')
print(f'Diagonal FCIM loss/gradient errors: {fcim_loss_error:.3e} / {fcim_gradient_error:.3e}')
print('PASS: n=6 symbolic, direct-CSR, fused, aggregate, Chebyshev, and FCIM paths agree.')

n=6: 33 genuine PennyLane operators
Dense term payload:  2.062 MiB
Sparse term payload: 49.629 KiB
Term-storage reduction: 42.56x
Stored numeric entries: 135,168 dense vs 2,112 sparse
Entry-count reduction: 64x
Hamiltonian max error: 0.000e+00
Direct CSR max error: 0.000e+00
Fused action max error: 3.334e-16
Loss error: 2.220e-16
Gradient max error: 2.776e-17
Aggregate loss/gradient errors: 2.220e-16 / 8.327e-17
Aggregate-state memory reduction (64 states): 32.00x
Chebyshev loss/gradient errors: 1.385e-10 / 2.795e-09
Adaptive Chebyshev degree (1e-8 tolerance): 24
Diagonal FCIM loss/gradient errors: 0.000e+00 / 2.776e-17
PASS: n=6 symbolic, direct-CSR, fused, aggregate, Chebyshev, and FCIM paths agree.


In [ ]:
plot(history_q, history_c, n)

In [ ]:
# to read data from csv instead of re-generating
n = 6
csv_filename = f"outputs/logloss_{n}qubit_heis_fcim.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)